# DIFUSCO: ER-[700-800] MIS 数据集完整复现 Notebook

本 Notebook 提供在 **ER-[700-800] 随机图** 上复现 **最大独立集 (MIS)** 问题的完整流程。

论文: [DIFUSCO: Graph-based Diffusion Solvers for Combinatorial Optimization](https://arxiv.org/abs/2302.08224) (NeurIPS 2023)

**核心配置**:
- 任务: MIS (Maximum Independent Set)
- 数据集: ER-[700-800] 随机图 (Erdős–Rényi, p=0.15)
- 扩散模型: Gaussian Diffusion
- 预训练检查点: `mis_er_gaussian.ckpt`

**流程概览**:
1. 环境准备与验证
2. ER-[700-800] 数据生成
3. 训练数据标注 (KaMIS 求解器)
4. 模型训练 (Gaussian Diffusion)
5. 使用预训练 checkpoint 进行评估 (贪心解码)
6. 使用预训练 checkpoint 进行评估 (4x 并行采样)
7. 结果分析


---
## 1. 环境准备与验证

### 1.1 安装依赖

请先按照仓库 README 创建 conda 环境：
```bash
conda env create -f environment.yml
conda activate difusco
```

**注意**: MIS 实验不需要编译 Cython 模块（那是 TSP 专用的）。


In [ ]:
import os
import subprocess
import textwrap
from pathlib import Path

# ---- 推断仓库根目录 ----
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "difusco").exists() and (REPO_ROOT.parent / "difusco").exists():
    REPO_ROOT = REPO_ROOT.parent
print(f"当前工作目录: {Path.cwd()}")
print(f"仓库根目录:   {REPO_ROOT}")


def run_cmd(cmd: str, check: bool = True):
    """Run a shell command and print stdout/stderr.

    Args:
        cmd: Shell command string to execute.
        check: Raise RuntimeError when command exits with non-zero code.

    Returns:
        subprocess.CompletedProcess returned by subprocess.run.
    """
    print("\n>>>", cmd)
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"命令执行失败 (code={result.returncode}): {cmd}")
    return result


In [ ]:
# 验证关键依赖
run_cmd("python --version", check=False)
run_cmd("python -c 'import torch; print(\"PyTorch version:\", torch.__version__); "
        "print(\"CUDA available:\", torch.cuda.is_available()); "
        "print(\"CUDA device count:\", torch.cuda.device_count())'", check=False)
run_cmd("python -c 'import pytorch_lightning as pl; print(\"PyTorch Lightning version:\", pl.__version__)'", check=False)
run_cmd("python -c 'import torch_geometric; print(\"PyG version:\", torch_geometric.__version__)'", check=False)
run_cmd("python -c 'import networkx; print(\"NetworkX version:\", networkx.__version__)'", check=False)
run_cmd("python -c 'import scipy; print(\"SciPy version:\", scipy.__version__)'", check=False)


---
## 2. 路径配置

请根据你的实际环境修改以下路径。所有路径均使用 **绝对路径**。


In [ ]:
# ============================================================
#  请根据实际环境修改以下路径
# ============================================================

# 存储路径（模型输出、日志、W&B 等）
STORAGE_PATH = "/your/storage/path"

# ER 数据目录（将在下一步生成）
ER_DATA_DIR = "/your/data_er"                         # ER 数据根目录
ER_TRAIN_DIR = f"{ER_DATA_DIR}/train"                 # 训练图文件目录
ER_TRAIN_ANNOTATIONS = f"{ER_DATA_DIR}/train_annotations"  # 训练标注目录
ER_VALID_DIR = f"{ER_DATA_DIR}/validation"            # 验证集目录
ER_TEST_DIR = f"{ER_DATA_DIR}/test"                   # 测试集目录

# 预训练检查点路径
CKPT_PATH = "/your/path/to/mis_er_gaussian.ckpt"

# GPU 设置
CUDA_VISIBLE_DEVICES = "0"  # 单 GPU 设置，多 GPU 改为 "0,1,2,3" 等

# ============================================================
#  环境变量设置
# ============================================================
os.environ["PYTHONPATH"] = f"{REPO_ROOT}:{os.environ.get('PYTHONPATH', '')}"
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

print("STORAGE_PATH:", STORAGE_PATH)
print("ER_DATA_DIR:", ER_DATA_DIR)
print("CKPT_PATH:", CKPT_PATH)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("PYTHONPATH:", os.environ["PYTHONPATH"])


---
## 3. ER-[700-800] 数据生成

使用 `mis-benchmark-framework` 中的数据生成工具，生成 ER (Erdős–Rényi) 随机图。

### 参数说明
- `--model er`: 使用 Erdős–Rényi 模型
- `--min_n 700 --max_n 800`: 节点数在 700 到 800 之间
- `--er_p 0.15`: 边概率为 0.15
- `--num_graphs`: 生成图的数量

论文中训练使用 163840 张图，这里提供完整配置和小规模测试配置。


In [ ]:
# 创建数据目录
os.makedirs(ER_TRAIN_DIR, exist_ok=True)
os.makedirs(ER_TRAIN_ANNOTATIONS, exist_ok=True)
os.makedirs(ER_VALID_DIR, exist_ok=True)
os.makedirs(ER_TEST_DIR, exist_ok=True)
print("数据目录已创建")


In [ ]:
# ============================================================
#  3.1 生成训练集 ER-[700-800] 图
# ============================================================
# 论文使用 163840 张图，可根据资源调整 --num_graphs
# 小规模测试可先设 --num_graphs 100

NUM_TRAIN_GRAPHS = 163840  # 论文完整配置
# NUM_TRAIN_GRAPHS = 100  # 小规模测试（取消注释此行）

gen_train_cmd = textwrap.dedent(f"""
cd {REPO_ROOT / 'data' / 'mis-benchmark-framework'} && \
python -u main.py gendata \
    random \
    None \
    {ER_TRAIN_DIR} \
    --model er \
    --min_n 700 \
    --max_n 800 \
    --num_graphs {NUM_TRAIN_GRAPHS} \
    --er_p 0.15
""").strip()

print("=== 训练集数据生成命令 ===")
print(gen_train_cmd)
# 取消下行注释来执行（耗时较长）
# run_cmd(gen_train_cmd)


In [ ]:
# ============================================================
#  3.2 生成验证集
# ============================================================
NUM_VALID_GRAPHS = 500  # 验证集大小，可调整

gen_valid_cmd = textwrap.dedent(f"""
cd {REPO_ROOT / 'data' / 'mis-benchmark-framework'} && \
python -u main.py gendata \
    random \
    None \
    {ER_VALID_DIR} \
    --model er \
    --min_n 700 \
    --max_n 800 \
    --num_graphs {NUM_VALID_GRAPHS} \
    --er_p 0.15
""").strip()

print("=== 验证集数据生成命令 ===")
print(gen_valid_cmd)
# run_cmd(gen_valid_cmd)


In [ ]:
# ============================================================
#  3.3 生成测试集
# ============================================================
# 论文中测试集来自 DIMESTeam/DIMES
# 如果没有现成数据，也可用以下方式自行生成

NUM_TEST_GRAPHS = 500  # 测试集大小，可调整

gen_test_cmd = textwrap.dedent(f"""
cd {REPO_ROOT / 'data' / 'mis-benchmark-framework'} && \
python -u main.py gendata \
    random \
    None \
    {ER_TEST_DIR} \
    --model er \
    --min_n 700 \
    --max_n 800 \
    --num_graphs {NUM_TEST_GRAPHS} \
    --er_p 0.15
""").strip()

print("=== 测试集数据生成命令 ===")
print(gen_test_cmd)
# run_cmd(gen_test_cmd)


---
## 4. 训练数据标注 (KaMIS 求解器)

生成训练标注需要使用 KaMIS 求解器求解每个图的最大独立集。
标注文件以 `_unweighted.result` 结尾，每行对应一个节点的标签 (0/1)。

### 前提
- 需要安装 KaMIS 求解器（参考 `data/mis-benchmark-framework/setup_bm_env.sh`）
- 需要创建 GPU 管理文件夹（用于 KaMIS 的任务管理）


In [ ]:
# 设置 GPU 管理文件夹（KaMIS 求解器需要）
os.makedirs("/tmp/gpus", exist_ok=True)
Path("/tmp/gpus/.lock").touch(exist_ok=True)
Path("/tmp/gpus/0.gpu").touch(exist_ok=True)
print("GPU 管理文件夹已设置")


In [ ]:
# ============================================================
#  4.1 使用 KaMIS 对训练数据生成标注
# ============================================================

solve_cmd = textwrap.dedent(f"""
cd {REPO_ROOT / 'data' / 'mis-benchmark-framework'} && \
python -u main.py \
    solve \
    kamis \
    {ER_TRAIN_DIR} \
    {ER_TRAIN_ANNOTATIONS} \
    --time_limit 60
""").strip()

print("=== KaMIS 求解命令 ===")
print(solve_cmd)
# run_cmd(solve_cmd)


In [ ]:
# ============================================================
#  4.2 验证数据生成结果
# ============================================================
import glob

train_graphs = glob.glob(os.path.join(ER_TRAIN_DIR, "*.gpickle"))
train_labels = glob.glob(os.path.join(ER_TRAIN_ANNOTATIONS, "*_unweighted.result"))
valid_graphs = glob.glob(os.path.join(ER_VALID_DIR, "*.gpickle"))
test_graphs = glob.glob(os.path.join(ER_TEST_DIR, "*.gpickle"))

print(f"训练集图文件数: {len(train_graphs)}")
print(f"训练集标注文件数: {len(train_labels)}")
print(f"验证集图文件数: {len(valid_graphs)}")
print(f"测试集图文件数: {len(test_graphs)}")

if len(train_graphs) > 0:
    print(f"\n示例训练图文件: {train_graphs[0]}")
if len(train_labels) > 0:
    print(f"示例标注文件:   {train_labels[0]}")


---
## 5. 模型训练

在 ER-[700-800] 数据集上训练 MIS Gaussian Diffusion 模型。

### 关键参数
| 参数 | 值 | 说明 |
|------|------|------|
| `--task` | `mis` | 最大独立集任务 |
| `--diffusion_type` | `gaussian` | 高斯扩散模型 |
| `--learning_rate` | `0.0002` | 学习率 |
| `--weight_decay` | `0.0001` | 权重衰减 |
| `--lr_scheduler` | `cosine-decay` | 余弦退火学习率调度 |
| `--batch_size` | `4` | 批大小 |
| `--num_epochs` | `50` | 训练轮数 |
| `--inference_schedule` | `cosine` | 推理使用余弦调度 |
| `--inference_diffusion_steps` | `50` | 推理扩散步数 |
| `--use_activation_checkpoint` | 启用 | 激活检查点（节省显存） |

**注意**: 
- 如果已有预训练 checkpoint `mis_er_gaussian.ckpt`，可直接跳到第 6 节进行评估。
- 完整训练需要多 GPU (论文使用 8 GPU) 和较长时间。


In [ ]:
# ============================================================
#  生成 W&B Run ID
# ============================================================
import wandb
WANDB_RUN_ID = wandb.util.generate_id()
os.environ["WANDB_RUN_ID"] = WANDB_RUN_ID
print(f"WANDB_RUN_ID: {WANDB_RUN_ID}")


In [ ]:
# ============================================================
#  5.1 训练命令
# ============================================================

train_cmd = textwrap.dedent(f"""
cd {REPO_ROOT} && \
python -u difusco/train.py \
  --task mis \
  --wandb_logger_name mis_diffusion_graph_gaussian_er \
  --diffusion_type gaussian \
  --do_train \
  --do_test \
  --learning_rate 0.0002 \
  --weight_decay 0.0001 \
  --lr_scheduler cosine-decay \
  --storage_path {STORAGE_PATH} \
  --training_split {ER_TRAIN_DIR}/*gpickle \
  --training_split_label_dir {ER_TRAIN_ANNOTATIONS}/ \
  --validation_split {ER_VALID_DIR}/*gpickle \
  --test_split {ER_TEST_DIR}/*gpickle \
  --batch_size 4 \
  --num_epochs 50 \
  --validation_examples 8 \
  --inference_schedule cosine \
  --inference_diffusion_steps 50 \
  --use_activation_checkpoint
""").strip()

print("=== 训练命令 ===")
print(train_cmd)
print()
print("提示: 取消下行注释开始训练（耗时很长，建议使用多GPU）")
# run_cmd(train_cmd)


---
## 6. 使用预训练 Checkpoint 进行评估

如果你已拥有预训练的 `mis_er_gaussian.ckpt`，可直接使用以下命令进行评估。

预训练 checkpoint 可从 [Google Drive](https://drive.google.com/drive/folders/1IjaWtkqTAs7lwtFZ24lTRspE0h1N6sBH?usp=sharing) 下载。


In [ ]:
# 验证 checkpoint 文件存在
if os.path.isfile(CKPT_PATH):
    file_size_mb = os.path.getsize(CKPT_PATH) / (1024 * 1024)
    print(f"Checkpoint 文件已找到: {CKPT_PATH}")
    print(f"文件大小: {file_size_mb:.2f} MB")
else:
    print(f"警告: Checkpoint 文件不存在: {CKPT_PATH}")
    print("请确认路径正确或从 Google Drive 下载预训练模型")


### 6.1 贪心解码评估 (Greedy Decoding)

贪心解码是最基础的评估方式：
- `--parallel_sampling 1` (默认): 每个图只生成一个解
- `--sequential_sampling 1` (默认): 不进行序贯采样


In [ ]:
# ============================================================
#  6.1 贪心解码评估
# ============================================================

eval_greedy_cmd = textwrap.dedent(f"""
cd {REPO_ROOT} && \
python -u difusco/train.py \
  --task mis \
  --wandb_logger_name mis_diffusion_graph_gaussian_er_test_greedy \
  --diffusion_type gaussian \
  --do_test \
  --learning_rate 0.0002 \
  --weight_decay 0.0001 \
  --lr_scheduler cosine-decay \
  --storage_path {STORAGE_PATH} \
  --training_split {ER_TRAIN_DIR}/*gpickle \
  --training_split_label_dir {ER_TRAIN_ANNOTATIONS}/ \
  --validation_split {ER_VALID_DIR}/*gpickle \
  --test_split {ER_TEST_DIR}/*gpickle \
  --batch_size 4 \
  --num_epochs 50 \
  --validation_examples 8 \
  --inference_schedule cosine \
  --inference_diffusion_steps 50 \
  --use_activation_checkpoint \
  --ckpt_path {CKPT_PATH} \
  --resume_weight_only
""").strip()

print("=== 贪心解码评估命令 ===")
print(eval_greedy_cmd)
print()
print("提示: 取消下行注释开始评估")
# run_cmd(eval_greedy_cmd)


### 6.2 并行采样评估 (4x Parallel Sampling)

并行采样可以提升求解质量：
- `--parallel_sampling 4`: 对每个图同时生成 4 个候选解
- 自动选取最优解 (最大独立集大小最大的)

这是论文 `reproducing_scripts.md` 中给出的 ER 数据集评估命令。


In [ ]:
# ============================================================
#  6.2 并行采样 (4x) 评估
# ============================================================

eval_parallel_cmd = textwrap.dedent(f"""
cd {REPO_ROOT} && \
python -u difusco/train.py \
  --task mis \
  --wandb_logger_name mis_diffusion_graph_gaussian_er_test_parallel4 \
  --diffusion_type gaussian \
  --do_test \
  --learning_rate 0.0002 \
  --weight_decay 0.0001 \
  --lr_scheduler cosine-decay \
  --storage_path {STORAGE_PATH} \
  --training_split {ER_TRAIN_DIR}/*gpickle \
  --training_split_label_dir {ER_TRAIN_ANNOTATIONS}/ \
  --validation_split {ER_VALID_DIR}/*gpickle \
  --test_split {ER_TEST_DIR}/*gpickle \
  --batch_size 4 \
  --num_epochs 50 \
  --validation_examples 8 \
  --inference_schedule cosine \
  --inference_diffusion_steps 50 \
  --parallel_sampling 4 \
  --use_activation_checkpoint \
  --ckpt_path {CKPT_PATH} \
  --resume_weight_only
""").strip()

print("=== 并行采样 (4x) 评估命令 ===")
print(eval_parallel_cmd)
print()
print("提示: 取消下行注释开始评估")
# run_cmd(eval_parallel_cmd)


### 6.3 序贯采样评估 (Sequential Sampling)

序贯采样在 batch_size=1 时使用，依次生成多个候选解再选最优：
- `--sequential_sampling 4`: 对每个图先后生成 4 个候选解
- 适用于显存受限的情况


In [ ]:
# ============================================================
#  6.3 序贯采样 (4x) 评估（可选）
# ============================================================

eval_sequential_cmd = textwrap.dedent(f"""
cd {REPO_ROOT} && \
python -u difusco/train.py \
  --task mis \
  --wandb_logger_name mis_diffusion_graph_gaussian_er_test_sequential4 \
  --diffusion_type gaussian \
  --do_test \
  --learning_rate 0.0002 \
  --weight_decay 0.0001 \
  --lr_scheduler cosine-decay \
  --storage_path {STORAGE_PATH} \
  --training_split {ER_TRAIN_DIR}/*gpickle \
  --training_split_label_dir {ER_TRAIN_ANNOTATIONS}/ \
  --validation_split {ER_VALID_DIR}/*gpickle \
  --test_split {ER_TEST_DIR}/*gpickle \
  --batch_size 4 \
  --num_epochs 50 \
  --validation_examples 8 \
  --inference_schedule cosine \
  --inference_diffusion_steps 50 \
  --sequential_sampling 4 \
  --use_activation_checkpoint \
  --ckpt_path {CKPT_PATH} \
  --resume_weight_only
""").strip()

print("=== 序贯采样 (4x) 评估命令 ===")
print(eval_sequential_cmd)
print()
print("提示: 取消下行注释开始评估")
# run_cmd(eval_sequential_cmd)


---
## 7. 结果分析

评估完成后，W&B 会记录以下关键指标：

| 指标 | 说明 |
|------|------|
| `test/solved_cost` | 模型求解的 MIS 大小（越大越好） |
| `test/gt_cost` | 真实最优解的 MIS 大小 |
| `val/solved_cost` | 验证集上模型求解的 MIS 大小 |

**解读**:
- `solved_cost` 越接近 `gt_cost` 说明模型求解质量越高
- 并行采样 (4x) 通常比贪心解码结果更好
- 论文中 ER-[700-800] 数据集的表现请参考论文 Table 2


In [ ]:
# ============================================================
#  7.1 查看 W&B 日志（可选）
# ============================================================
# 训练和评估日志保存在 W&B 中
# 可以通过 W&B 网页端查看详细指标

print("请在 W&B 网页端查看完整的训练和评估指标:")
print("https://wandb.ai/")
print()
print(f"模型日志保存在: {STORAGE_PATH}/models/")


In [ ]:
# ============================================================
#  7.2 手动加载 checkpoint 并检查模型结构
# ============================================================
import torch

if os.path.isfile(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    print("Checkpoint 包含的键:")
    for k in ckpt.keys():
        if k == "state_dict":
            print(f"  {k}: {len(ckpt[k])} 个参数")
        elif k == "hyper_parameters":
            print(f"  {k}: {ckpt[k]}")
        else:
            print(f"  {k}: {type(ckpt[k]).__name__}")

    if "state_dict" in ckpt:
        total_params = sum(p.numel() for p in ckpt["state_dict"].values())
        print(f"\n模型总参数量: {total_params:,}")
else:
    print(f"Checkpoint 不存在: {CKPT_PATH}")


In [ ]:
# ============================================================
#  7.3 检查单个图文件（可选）
# ============================================================
import glob

test_files = sorted(glob.glob(os.path.join(ER_TEST_DIR, "*.gpickle")))
if len(test_files) > 0:
    try:
        import pickle5 as pickle
    except ImportError:
        import pickle

    with open(test_files[0], "rb") as f:
        graph = pickle.load(f)

    print(f"图文件: {test_files[0]}")
    print(f"节点数: {graph.number_of_nodes()}")
    print(f"边数:   {graph.number_of_edges()}")
    print(f"密度:   {graph.number_of_edges() / (graph.number_of_nodes() * (graph.number_of_nodes() - 1) / 2):.4f}")

    node_labels = [n[1] for n in graph.nodes(data='label')]
    if node_labels[0] is not None:
        import numpy as np
        labels = np.array(node_labels)
        print(f"MIS 大小 (标签): {labels.sum()}")
    else:
        print("该图文件没有节点标签")
else:
    print("测试集目录为空，请先生成数据")


---
## 8. 常见问题排查

### Q1: 找不到数据文件
检查 `--training_split`, `--validation_split`, `--test_split` 路径是否正确。
注意 glob 模式 `*gpickle` 需要目录中确实存在 `.gpickle` 文件。

### Q2: 显存不足 (CUDA OOM)
- 减小 `--batch_size`（最小可设为 1）
- 添加 `--use_activation_checkpoint` 标志
- 使用 `--fp16` 开启混合精度训练
- 减少 `--parallel_sampling` 数量

### Q3: KaMIS 求解器安装失败
参考 `data/mis-benchmark-framework/setup_bm_env.sh` 安装脚本。
或者直接从 [DIMESTeam/DIMES](https://github.com/DIMESTeam/DIMES) 下载已标注数据。

### Q4: W&B 登录问题
```python
import wandb
wandb.login()
```
或设置环境变量 `WANDB_API_KEY`，或使用 `WANDB_MODE=offline` 离线模式。

### Q5: 训练路径中的 `--training_split_label_dir`
这个参数指向 KaMIS 生成的标注文件目录。
注意：该路径会与 `--storage_path` 拼接，参见 `pl_mis_model.py` 第 25 行：
```python
data_label_dir = os.path.join(self.args.storage_path, self.args.training_split_label_dir)
```

### Q6: 多 GPU 训练
设置 `CUDA_VISIBLE_DEVICES` 环境变量：
```python
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"  # 8 GPU
```


---
## 9. 完整复现步骤总结

### 仅评估 (使用预训练 checkpoint)

1. 准备环境 (`conda env create -f environment.yml`)
2. 下载预训练 checkpoint `mis_er_gaussian.ckpt`
3. 准备测试数据 (ER-[700-800] 图文件)
4. 运行第 6.1 节 (贪心解码) 或第 6.2 节 (并行采样)

### 从头训练 + 评估

1. 准备环境
2. 生成训练数据 (第 3 节)
3. 标注训练数据 (第 4 节)
4. 训练模型 (第 5 节)
5. 评估模型 (第 6 节)
